In [5]:
import os, random, numpy as np, pandas as pd
import torch, torch.nn as nn, torchaudio
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


In [6]:
# Load metadata, binary labels: control=0, dysfluent=1
df = pd.read_csv(os.path.join(BASE_DIR, "data", "metadata.csv"))
df["label_id"] = (df["label"] == "dysfluent").astype(int)
df["abs_path"] = df["file_path"].apply(lambda p: os.path.join(BASE_DIR, p))

# Remove mild from dataset
df = df[df["severity"] != "mild"].reset_index(drop=True)
print(f"After removing mild: {len(df)} samples")
print(df["severity"].value_counts().to_string())

train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label_id"])
print(f"\nTrain: {len(train_df)} ({train_df['label_id'].mean():.1%} dysfluent)")
print(f"Test:  {len(test_df)} ({test_df['label_id'].mean():.1%} dysfluent)")

After removing mild: 834 samples
severity
moderate    327
none        259
severe      248

Train: 667 (69.0% dysfluent)
Test:  167 (68.9% dysfluent)


In [7]:
TARGET_SR = 16000

# ── Waveform augmentations (applied before wav2vec2) ──
from scipy.signal import fftconvolve

def augment_waveform(wav):
    """Apply random waveform augmentations. Input/output: 1-D tensor."""
    # Speed perturbation (0.9-1.1x) via linear interpolation (fast)
    if random.random() < 0.5:
        factor = random.uniform(0.9, 1.1)
        new_len = int(len(wav) / factor)
        wav = torch.nn.functional.interpolate(wav.view(1, 1, -1), size=new_len, mode="linear", align_corners=False).squeeze()
    # Additive noise (SNR 10-20 dB)
    if random.random() < 0.5:
        snr_db = random.uniform(10, 20)
        sig_pow = wav.pow(2).mean()
        noise = torch.randn_like(wav) * (sig_pow / (10 ** (snr_db / 10))).sqrt()
        wav = wav + noise
    # Volume perturbation (-6 to +6 dB)
    if random.random() < 0.5:
        wav = wav * (10 ** (random.uniform(-6, 6) / 20))
    # Room impulse response (RT60 0.2-0.8s)
    if random.random() < 0.3:
        rt60 = random.uniform(0.2, 0.8)
        n = int(TARGET_SR * rt60)
        t = torch.arange(n, dtype=torch.float32) / TARGET_SR
        rir = torch.randn(n) * torch.exp(-3.0 * t / rt60)
        rir = rir / rir.abs().max()
        out = fftconvolve(wav.numpy(), rir.numpy(), mode="full")[:len(wav)]
        wav = torch.from_numpy(out.astype(np.float32))
    return wav

class SpeechDataset(Dataset):
    def __init__(self, dataframe, processor, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav, sr = torchaudio.load(row["abs_path"])
        if sr != TARGET_SR:
            wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
        wav = wav.squeeze(0)
        if self.augment:
            wav = augment_waveform(wav)
        inputs = self.processor(wav, sampling_rate=TARGET_SR, return_tensors="pt", padding=False)
        return inputs.input_values.squeeze(0), row["label_id"]

def collate_fn(batch):
    wavs, labels = zip(*batch)
    max_len = max(w.shape[0] for w in wavs)
    padded = torch.zeros(len(wavs), max_len)
    mask = torch.zeros(len(wavs), max_len, dtype=torch.bool)
    for i, w in enumerate(wavs):
        padded[i, :w.shape[0]] = w
        mask[i, :w.shape[0]] = True
    return padded, mask, torch.tensor(labels, dtype=torch.long)

In [8]:
# Load frozen wav2vec2
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(DEVICE)
w2v.eval()
for p in w2v.parameters():
    p.requires_grad = False

# Single linear layer: 768 -> 2
head = nn.Linear(768, 2).to(DEVICE)
print(f"Learnable params: {sum(p.numel() for p in head.parameters())}")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 839.08it/s, Materializing param=masked_spec_embed]                                             
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Learnable params: 1538


In [9]:
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3

train_loader = DataLoader(SpeechDataset(train_df, processor, augment=True), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(SpeechDataset(test_df, processor, augment=False), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = torch.optim.Adam(head.parameters(), lr=LR)
n_dys = (train_df["label_id"] == 1).sum()
n_ctrl = (train_df["label_id"] == 0).sum()
weights = torch.tensor([n_dys / n_ctrl, 1.0], dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)

# SpecAugment (applied on wav2vec2 hidden states during training)
freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=10)
time_mask = torchaudio.transforms.TimeMasking(time_mask_param=15)

print(f"Class weights: control={weights[0]:.2f}, dysfluent={weights[1]:.2f}")

Class weights: control=2.22, dysfluent=1.00


In [10]:
from tqdm.auto import tqdm

for epoch in range(EPOCHS):
    head.train()
    total_loss, correct, total = 0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False)
    for wavs, mask, labels in pbar:
        wavs, mask, labels = wavs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)

        with torch.no_grad():
            out = w2v(wavs, attention_mask=mask.float())
            hidden = out.last_hidden_state  # (B, T_out, 768)

        # SpecAugment on hidden states (treat as (B, 768, T) "spectrogram")
        hidden_t = hidden.transpose(1, 2)  # (B, 768, T)
        hidden_t = freq_mask(hidden_t)
        hidden_t = time_mask(hidden_t)
        hidden = hidden_t.transpose(1, 2)  # (B, T, 768)

        # Build output-length mask + mean pool
        input_lengths = mask.sum(1)
        output_lengths = w2v._get_feat_extract_output_lengths(input_lengths).long()
        out_mask = torch.arange(hidden.size(1), device=DEVICE).unsqueeze(0) < output_lengths.unsqueeze(1)
        pooled = (hidden * out_mask.unsqueeze(-1)).sum(1) / output_lengths.unsqueeze(1)

        logits = head(pooled)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.3f}")

    print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={total_loss/total:.4f}  acc={correct/total:.3f}")

Epoch  1/20  loss=0.5727  acc=0.700


Epoch  2/20  loss=0.5493  acc=0.736


Epoch  3/20  loss=0.5137  acc=0.771


Epoch  4/20  loss=0.5191  acc=0.777


Epoch  5/20  loss=0.5083  acc=0.753


...

Epoch 16/20  loss=0.4287  acc=0.784


Epoch 17/20  loss=0.4371  acc=0.799


Epoch 18/20  loss=0.4176  acc=0.817


Epoch 19/20  loss=0.4324  acc=0.786


Epoch 20/20  loss=0.4189  acc=0.823


In [11]:
# Evaluate
head.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for wavs, mask, labels in test_loader:
        wavs, mask = wavs.to(DEVICE), mask.to(DEVICE)
        hidden = w2v(wavs, attention_mask=mask.float()).last_hidden_state
        input_lengths = mask.sum(1)
        output_lengths = w2v._get_feat_extract_output_lengths(input_lengths).long()
        out_mask = torch.arange(hidden.size(1), device=DEVICE).unsqueeze(0) < output_lengths.unsqueeze(1)
        pooled = (hidden * out_mask.unsqueeze(-1)).sum(1) / output_lengths.unsqueeze(1)
        preds = head(pooled).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print(classification_report(all_labels, all_preds, target_names=["control", "dysfluent"]))

              precision    recall  f1-score   support

     control       0.57      0.73      0.64        52
   dysfluent       0.91      0.83      0.87       170

    accuracy                           0.81       222
   macro avg       0.74      0.78      0.75       222
weighted avg       0.83      0.81      0.81       222



In [11]:
# Per-group accuracy (control, mild, moderate, severe) on full dataset
head.eval()
group_results = {s: {"correct": 0, "total": 0} for s in ["none", "mild", "moderate", "severe"]}

for severity, group_df in df.groupby("severity"):
    loader = DataLoader(SpeechDataset(group_df, processor), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
    for wavs, mask, labels in loader:
        wavs, mask, labels = wavs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            hidden = w2v(wavs, attention_mask=mask.float()).last_hidden_state
            input_lengths = mask.sum(1)
            output_lengths = w2v._get_feat_extract_output_lengths(input_lengths).long()
            out_mask = torch.arange(hidden.size(1), device=DEVICE).unsqueeze(0) < output_lengths.unsqueeze(1)
            pooled = (hidden * out_mask.unsqueeze(-1)).sum(1) / output_lengths.unsqueeze(1)
            preds = head(pooled).argmax(1)
        group_results[severity]["correct"] += (preds == labels).sum().item()
        group_results[severity]["total"] += labels.size(0)

labels_map = {"none": "control", "mild": "mild", "moderate": "moderate", "severe": "severe"}
for sev in ["none", "mild", "moderate", "severe"]:
    r = group_results[sev]
    acc = r["correct"] / r["total"] if r["total"] > 0 else 0
    print(f"{labels_map[sev]:>10s}  ({r['total']:3d} clips)  acc={acc:.3f}")

   control  (259 clips)  acc=0.340
      mild  (  0 clips)  acc=0.000
  moderate  (327 clips)  acc=0.893
    severe  (248 clips)  acc=0.972


In [12]:
# Save weights
save_dir = os.path.join(BASE_DIR, "data", "models")
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "baseline_logreg2.pt")
torch.save(head.state_dict(), save_path)
print(f"Saved → {save_path}")

Saved → /data/liharrison/lvsim/data/models/baseline_logreg2.pt


In [13]:
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
w2v = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").eval()
head = torch.nn.Linear(768, 2)
head.load_state_dict(torch.load("/data/liharrison/lvsim/data/models/baseline_logreg2.pt", weights_only=True))
head.eval()

wav, sr = torchaudio.load("/data/liharrison/lvsim/data/real/segmentedcc-20260227T072422Z-1-001/segmentedcc/02-1_PAR_002.wav")
wav = torchaudio.functional.resample(wav, sr, 16000).squeeze(0)
inputs = processor(wav, sampling_rate=16000, return_tensors="pt")

with torch.no_grad():
    hidden = w2v(**inputs).last_hidden_state
    pooled = hidden.mean(dim=1)  # (1, 768)
    logits = head(pooled)
    pred = logits.argmax(1).item()

print("dysfluent" if pred == 1 else "control", f"({torch.softmax(logits, 1).squeeze()})")

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 842.27it/s, Materializing param=masked_spec_embed]                                            
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dysfluent (tensor([0.0885, 0.9115]))


In [14]:
# Run on real data folders
import glob

REAL_DIR = os.path.join(BASE_DIR, "data", "real")
groups = {
    "lvPPA":   glob.glob(os.path.join(REAL_DIR, "*lvPPA*", "**", "*.wav"), recursive=True),
    "JHU":     glob.glob(os.path.join(REAL_DIR, "*jhu*", "**", "*.wav"), recursive=True),
    "Control": glob.glob(os.path.join(REAL_DIR, "*cc*", "**", "*.wav"), recursive=True),
}

head.to(DEVICE).eval()
w2v.to(DEVICE).eval()

for name, files in groups.items():
    preds = []
    for f in files:
        wav, sr = torchaudio.load(f)
        wav = torchaudio.functional.resample(wav, sr, 16000).mean(0)
        inputs = processor(wav, sampling_rate=16000, return_tensors="pt").input_values.to(DEVICE)
        with torch.no_grad():
            hidden = w2v(inputs).last_hidden_state
            pred = head(hidden.mean(1)).argmax(1).item()
        preds.append(pred)
    n_dys = sum(preds)
    n_ctrl = len(preds) - n_dys
    print(f"{name:>8s}  ({len(files):3d} clips)  →  control={n_ctrl}  dysfluent={n_dys}")

   lvPPA  ( 89 clips)  →  control=0  dysfluent=89
     JHU  ( 74 clips)  →  control=0  dysfluent=74
 Control  (235 clips)  →  control=0  dysfluent=235


In [10]:
# ── Augmentation demo: save augmented versions of example clips ──
import torch, torchaudio
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve

TARGET_SR = 16000

def speed_perturb(wav, factor=None):
    """Speed perturbation (0.9-1.1x) via F.interpolate (fast)."""
    if factor is None:
        factor = random.uniform(0.9, 1.1)
    new_len = int(len(wav) / factor)
    out = F.interpolate(wav.view(1, 1, -1), size=new_len, mode="linear", align_corners=False).squeeze()
    return out, factor

def add_noise(wav, snr_db=None):
    """Additive Gaussian noise at random SNR (10-20 dB)."""
    if snr_db is None:
        snr_db = random.uniform(10, 20)
    signal_power = wav.pow(2).mean()
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = torch.randn_like(wav) * noise_power.sqrt()
    return wav + noise, snr_db

def volume_perturb(wav, gain_db=None):
    """Random volume change (-6 to +6 dB)."""
    if gain_db is None:
        gain_db = random.uniform(-6, 6)
    return wav * (10 ** (gain_db / 20)), gain_db

def simulate_rir(wav, sr, rt60=None):
    """Simple synthetic room impulse response via exponential decay."""
    if rt60 is None:
        rt60 = random.uniform(0.2, 0.8)
    n_samples = int(sr * rt60)
    t = torch.arange(n_samples, dtype=torch.float32) / sr
    rir = torch.randn(n_samples) * torch.exp(-3.0 * t / rt60)
    rir = rir / rir.abs().max()
    wav_np = wav.numpy()
    rir_np = rir.numpy()
    out = fftconvolve(wav_np, rir_np, mode="full")[:len(wav_np)]
    out = torch.from_numpy(out.astype(np.float32))
    return out / out.abs().max(), rt60

# ── Run on 2 example clips ──
examples = [
    os.path.join(BASE_DIR, "data", "dysfluent", "severe", "dys_severe_gt0_spk36_s000.wav"),
    os.path.join(BASE_DIR, "data", "control", "ctrl_gt0_spk36_s000.wav"),
]

DEMO_DIR = os.path.join(BASE_DIR, "data", "misc", "aug_demo")
os.makedirs(DEMO_DIR, exist_ok=True)

for path in examples:
    name = os.path.splitext(os.path.basename(path))[0]
    wav, sr = torchaudio.load(path)
    wav = torchaudio.functional.resample(wav, sr, TARGET_SR).squeeze(0)
    
    # Save original
    torchaudio.save(os.path.join(DEMO_DIR, f"{name}_original.wav"), wav.unsqueeze(0), TARGET_SR)
    
    # Speed
    aug, factor = speed_perturb(wav, factor=1.08)
    torchaudio.save(os.path.join(DEMO_DIR, f"{name}_speed_{factor:.2f}.wav"), aug.unsqueeze(0), TARGET_SR)
    
    # Noise
    aug, snr = add_noise(wav, snr_db=12)
    torchaudio.save(os.path.join(DEMO_DIR, f"{name}_noise_{snr:.0f}dB.wav"), aug.unsqueeze(0), TARGET_SR)
    
    # Volume
    aug, gain = volume_perturb(wav, gain_db=-4)
    torchaudio.save(os.path.join(DEMO_DIR, f"{name}_vol_{gain:+.0f}dB.wav"), aug.unsqueeze(0), TARGET_SR)
    
    # RIR
    aug, rt60 = simulate_rir(wav, TARGET_SR, rt60=0.5)
    torchaudio.save(os.path.join(DEMO_DIR, f"{name}_rir_{rt60:.1f}s.wav"), aug.unsqueeze(0), TARGET_SR)
    
    # SpecAugment — save before/after spectrogram image (feature-space aug, not invertible to audio)
    mel_transform = torchaudio.transforms.MelSpectrogram(sample_rate=TARGET_SR, n_mels=80, n_fft=512)
    spec_orig = mel_transform(wav.unsqueeze(0)).log2().clamp(min=-10)
    spec_aug = spec_orig.clone()
    spec_aug = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)(spec_aug)
    spec_aug = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)(spec_aug)
    spec_aug = torchaudio.transforms.TimeMasking(time_mask_param=30)(spec_aug)
    spec_aug = torchaudio.transforms.TimeMasking(time_mask_param=30)(spec_aug)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
    ax1.imshow(spec_orig.squeeze().numpy(), aspect="auto", origin="lower"); ax1.set_title("Original")
    ax2.imshow(spec_aug.squeeze().numpy(), aspect="auto", origin="lower"); ax2.set_title("SpecAugment")
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(os.path.join(DEMO_DIR, f"{name}_specaug.png"), dpi=150)
    plt.close(fig)
    
    print(f"{name}: saved 4 augmented WAVs + specaug image + original")

print(f"\nAll saved to {DEMO_DIR}/")

dys_severe_gt0_spk36_s000: saved 4 augmented WAVs + specaug image + original
ctrl_gt0_spk36_s000: saved 4 augmented WAVs + specaug image + original

All saved to /data/liharrison/lvsim/data/misc/aug_demo/


In [ ]:
# ── Diagnose: DataLoader vs GPU forward pass ──
import time

# Test 1: DataLoader only (CPU augmentation)
print("DataLoader only (CPU augmentation):")
t0 = time.time()
for i, (wavs, mask, labels) in enumerate(train_loader):
    if i >= 10:
        break
print(f"  10 batches in {time.time()-t0:.2f}s  ({(time.time()-t0)/10*1000:.0f} ms/batch)\n")

# Test 2: Full pipeline with augmentation
print("Full pipeline (load + augment + w2v + head):")
t0 = time.time()
for i, (wavs, mask, labels) in enumerate(train_loader):
    wavs, mask, labels = wavs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
    with torch.no_grad():
        hidden = w2v(wavs, attention_mask=mask.float()).last_hidden_state
    if i >= 10:
        break
torch.cuda.synchronize()
print(f"  10 batches in {time.time()-t0:.2f}s  ({(time.time()-t0)/10*1000:.0f} ms/batch)\n")

# Test 3: Without augmentation for comparison
noaug_loader = DataLoader(SpeechDataset(train_df, processor, augment=False), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
print("Full pipeline WITHOUT augmentation:")
t0 = time.time()
for i, (wavs, mask, labels) in enumerate(noaug_loader):
    wavs, mask, labels = wavs.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
    with torch.no_grad():
        hidden = w2v(wavs, attention_mask=mask.float()).last_hidden_state
    if i >= 10:
        break
torch.cuda.synchronize()
print(f"  10 batches in {time.time()-t0:.2f}s  ({(time.time()-t0)/10*1000:.0f} ms/batch)")